In [1]:
from itertools import chain
from ssl import PROTOCOL_TLS
from typing import Callable, Sequence, Dict
from __future__ import annotations

import inspect
import dataclasses
from typing import (
    Any, get_type_hints, get_origin, get_args, Union, Optional, Literal, Annotated
)
from to_json_schema.to_json_schema import SchemaBuilder

import paho.mqtt.client as mqtt
import numpy as np
import json

from ascribe_link.example import sphere_example

In [2]:
# Define the enumerated functions for data processing
def random_mesh(*args, **kwargs):
    # Implement processing logic here
    return np.random.rand(10, 3), np.random.randint(10, (10, 3))

In [3]:
# Define a dictionary mapping function names to implementations
function_map = {
    'sphere': sphere_example,
    'random_mesh': random_mesh
}


In [4]:
def validate_mesh(points, indices):
    # Check for non-finite vertices
    bad_points = np.array(points)[~np.isfinite(points).all(axis=1)]
    if len(bad_points):
        print("Bad points:", bad_points)
        raise ValueError("Mesh contains points with infinite values")


In [5]:
def on_message(client, userdata, message):
    topic = message.topic
    request_data = json.loads(message.payload)
    print(client, userdata, topic, request_data)
    match topic:
        case 'godot/processing_requests':
            function_name = request_data['function_name']
            args = request_data['args']
            kwargs = request_data['kwargs']
 
            # Call the corresponding function and serialize the result
            result = function_map[function_name](*args, **kwargs)
            result_data = {'vertices': list(chain.from_iterable(result[0])),
                           'indices': result[1]}

            # Validate before sending
            validate_mesh(result[0], result[1])

            # Publish the result to the processing responses topic
            client.publish("python/processing_responses", json.dumps(result_data))
        case 'godot/specimen_requests':
            function_names = list(function_map.keys())
            response = dict(names=function_names)
            client.publish("python/specimen_responses", json.dumps(response))


In [6]:
def on_connect(client, userdata, flags, reason_code, properties):
    print(f"Connected with result code {reason_code}")
    # Subscribing in on_connect() means that if we lose the connection and
    # reconnect then subscriptions will be renewed.
    client.subscribe("$SYS/#")

In [7]:
def serve(broker=None, port=1883, client=None, mesh_functions: Dict[str, Callable]=None):
    if mesh_functions:
        function_map.clear()
        function_map.update(mesh_functions)

    if client is None:
        client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
        client.connect(broker, port)

    # Subscribe to the processing requests topic
    client.subscribe("godot/processing_requests")

    # Subscribe to the specimen requests topic
    client.subscribe("godot/specimen_requests")

    # Set the callback for incoming messages
    client.on_message = on_message
    client.on_connect = on_connect

    # Start the MQTT loop
    client.loop_forever()

In [8]:
def some_func(arg1: int, arg2: bool, arg3: str):
    return 23

In [24]:
def create_schema(func: Callable):

    schema = {
        "$schema": None,
        "$id": None,
        "title": func.__name__,
        "type": "object",
        "properties": None
        
    }
    # set the name of the function
    schema['title'] = func.__name__
    # schema['type'] = "Object"
    # print(schema['title'])
    # print(inspect.signature(func))

    # based on the signature we can figure out the properties and type
    signature = inspect.signature(func)
    if signature.return_annotation != signature.empty:
        schema['type'] = signature.return_annotation
    # print(schema['type'])
    # set the arguments as the properties
    schema['properties'] = signature.parameters
    # print(schema['properties'])
    
    return schema
        
        
        
        
        
    
    
    
    
    

In [25]:
if __name__ == '__main__':

    print(create_schema(some_func))
    # Client setup
    # client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)  # , transport="websockets")
    # client.connect("vision.lbl.gov", 1883)
    # # print(create_schema(sphere_example))
    # # Start server
    # serve(client=client, mesh_functions={"Automated Thresholding":0, "Unsupervised ML":1, "Supervised ML":2})

{'$schema': None, '$id': None, 'title': 'some_func', 'type': 'object', 'properties': mappingproxy(OrderedDict({'arg1': <Parameter "arg1: 'int'">, 'arg2': <Parameter "arg2: 'bool'">, 'arg3': <Parameter "arg3: 'str'">}))}
